In [0]:
%python
from pyspark.sql.types import *
from pyspark.sql.functions import *

bronze_orders = spark.readStream.format("delta").table("streaming_project.ecommerce.bronze_orders")

silver_orders = (
    bronze_orders
    # Cast date string to timestamp
    .withColumn("order_date", to_timestamp(col("order_date")))
    # Extract date parts for partitioning
    .withColumn("order_year",  year(col("order_date")))
    .withColumn("order_month", month(col("order_date")))
    .withColumn("order_day",   dayofmonth(col("order_date")))
    # Add total price per line item
    .withColumn("total_price", col("quantity") * col("price") 
                if "price" in bronze_orders.columns 
                else lit(None).cast(DoubleType()))
    # Add ingestion timestamp
    .withColumn("ingested_at", current_timestamp())
    # Drop nulls on key fields
    .filter(col("order_id").isNotNull() & col("customer_id").isNotNull())
)

(
    silver_orders.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/silver_orders")
    .trigger(availableNow=True)
    .toTable("streaming_project.ecommerce.silver_orders")
)

print("✅ silver_orders written")

In [0]:
%python
from pyspark.sql.types import *
from pyspark.sql.functions import *

bronze_customers = spark.readStream.format("delta").table("streaming_project.ecommerce.bronze_customers")

silver_customers = (
    bronze_customers
    # Combine first + last name
    .withColumn("full_name", concat(initcap(col("first_name")), lit(" "), initcap(col("last_name"))))
    # Standardize city and street casing
    .withColumn("city",   initcap(col("city")))
    .withColumn("street", initcap(col("street")))
    # Validate email contains @
    .withColumn("is_valid_email", col("email").contains("@"))
    # Drop sensitive column
    .drop("password")
    # Add ingestion timestamp
    .withColumn("ingested_at", current_timestamp())
    # Drop nulls on key fields
    .filter(col("customer_id").isNotNull())
)

(
    silver_customers.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/silver_customers")
    .trigger(availableNow=True)
    .toTable("streaming_project.ecommerce.silver_customers")
)

print("✅ silver_customers written")

✅ silver_customers written


In [0]:
%python
from pyspark.sql.types import *
from pyspark.sql.functions import *

bronze_products = spark.readStream.format("delta").table("streaming_project.ecommerce.bronze_products")

silver_products = (
    bronze_products
    # Cast price to Decimal
    .withColumn("price", col("price").cast(DecimalType(10, 2)))
    # Standardize category casing
    .withColumn("category", initcap(col("category")))
    # Add price category bucket
    .withColumn("price_category",
        when(col("price") < 20,  "Budget")
        .when(col("price") < 100, "Mid-Range")
        .otherwise("Premium")
    )
    # Add rating category
    .withColumn("rating_category",
        when(col("rating_rate") >= 4.5, "Excellent")
        .when(col("rating_rate") >= 3.5, "Good")
        .when(col("rating_rate") >= 2.5, "Average")
        .otherwise("Poor")
    )
    # Add ingestion timestamp
    .withColumn("ingested_at", current_timestamp())
    # Drop nulls on key fields
    .filter(col("product_id").isNotNull())
)

(
    silver_products.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/streaming_project/ecommerce/stream_volume/checkpoints/silver_products")
    .trigger(availableNow=True)
    .toTable("streaming_project.ecommerce.silver_products")
)

print("✅ silver_products written")

✅ silver_products written
